In [ ]:
# Magnitude of the Sun from L2
# Alejandro S. Borlaff - NASA / Ames Research Center 
# a.s.borlaff@nasa.gov

import rosalia as rs

In [ ]:
mjd = 60825

observer = "@jwst"



In [ ]:
def earth_sun_moon_from_observer(mjd, observer, verbose=True):
    # verbose=True
    earth_hzjpl = rs.horizons.horizons_query(mjd, source="399", location=observer) # 399 : Earth
    sun_hzjpl = rs.horizons.horizons_query(mjd, source="@sun", location=observer) # Sun 
    moon_hzjpl = rs.horizons.horizons_query(mjd, source="301", location=observer) # 301 : Moon / Luna

    return({"earth_hzjpl": earth_hzjpl, "sun_hzjpl": sun_hzjpl, "moon_hzjpl": moon_hzjpl})


In [ ]:
import numpy as np
from tqdm import tqdm
mjd_list = np.linspace(59577+100, 59577+425, 1000)

earth_ra = np.zeros(mjd_list.shape)
earth_dec = np.zeros(mjd_list.shape)
earth_size = np.zeros(mjd_list.shape)

moon_ra = np.zeros(mjd_list.shape)
moon_dec = np.zeros(mjd_list.shape)
moon_size = np.zeros(mjd_list.shape)

sun_ra = np.zeros(mjd_list.shape)
sun_dec = np.zeros(mjd_list.shape)
sun_size = np.zeros(mjd_list.shape)

for i in tqdm(range(len(mjd_list))):

    mjd = mjd_list[i]
                   
    earth_sun_moon = earth_sun_moon_from_observer(mjd=mjd, observer="@jwst", verbose=True)

    earth_ra[i] = earth_sun_moon["earth_hzjpl"]["jpl_horizons_query"].ephemerides()["RA"][0]
    earth_dec[i] = earth_sun_moon["earth_hzjpl"]["jpl_horizons_query"].ephemerides()["DEC"][0]
    earth_size[i] = earth_sun_moon["earth_hzjpl"]["jpl_horizons_query"].ephemerides()["ang_width"][0]/60/60

    moon_ra[i] = earth_sun_moon["moon_hzjpl"]["jpl_horizons_query"].ephemerides()["RA"][0]
    moon_dec[i] = earth_sun_moon["moon_hzjpl"]["jpl_horizons_query"].ephemerides()["DEC"][0]
    moon_size[i] = earth_sun_moon["moon_hzjpl"]["jpl_horizons_query"].ephemerides()["ang_width"][0]/60/60

    sun_ra[i] = earth_sun_moon["sun_hzjpl"]["jpl_horizons_query"].ephemerides()["RA"][0]
    sun_dec[i] = earth_sun_moon["sun_hzjpl"]["jpl_horizons_query"].ephemerides()["DEC"][0]
    sun_size[i] = earth_sun_moon["sun_hzjpl"]["jpl_horizons_query"].ephemerides()["ang_width"][0]/60/60
    

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')
if True:
    for i in tqdm(range(len(mjd_list))):
        
        fig, ax = plt.subplots(figsize=(16,9)) # note we must use plt.subplots, not plt.subplot

        earth_color = "lightblue"
        sun_color = "gold"
        moon_color = "silver"
        sun_marker = u'$\u2609$' 
        moon_marker = u'$\u263D$' 
        earth_marker = u'$\u1F728$'
        
        ax.plot(earth_ra[:i], 
                   earth_dec[:i], 
                   color=earth_color, alpha=0.5, linestyle="-", linewidth=2)
        
        ax.plot(sun_ra[:i], 
                   sun_dec[:i], 
                   color=sun_color, alpha=0.5, linestyle="-", linewidth=2)
        
        ax.plot(moon_ra[:i], 
                   moon_dec[:i], 
                   color=moon_color, alpha=0.5, linestyle="-", linewidth=2)
        
        
        earth_circle = plt.Circle((earth_ra[i], 
                                   earth_dec[i]), 
                                   earth_size[i], 
                                   color=earth_color)
        ax.add_patch(earth_circle)
        
        sun_circle = plt.Circle((sun_ra[i], 
                                 sun_dec[i]), 
                                 sun_size[i], 
                                 color=sun_color)
        ax.add_patch(sun_circle)

        moon_circle = plt.Circle((moon_ra[i], 
                                  moon_dec[i]), 
                                  moon_size[i], 
                                   color=moon_color)
        ax.add_patch(moon_circle)
        ax.set_xlabel("Right Ascension (degrees)")
        ax.set_ylabel("Declination (degrees)")

        from astropy.time import Time

        isot = Time(mjd_list[i], format="mjd").isot
        
        ax.text(280, 80, isot)
        
        ax.set_xlim(0, 360)
        ax.set_ylim(-90, 90)

        plt.savefig("earth_moon_sun_from_L2_" + str(i).zfill(5) + ".png", dpi=100)
        plt.show()


In [ ]:
import os 
os.system("magick -delay 5 -loop 0 earth_moon_sun_from_L2_*.png earth_moon_sun_from_L2.gif")